In [ ]:
from collections import Counter, deque
from functools import lru_cache
import re
import json

class BPETokenizerSimple:
    def __init__(self):
        self.vocab= {}  #maps token ids to token_str
        self.inverse_vocab= {} #maps token str to toke ids
        self.bpe_merges ={} #dictionary of bpe merges
#           training text  desired vocab size    set of special tokens to be included
    def train(self, text, vocab_size, allowed_specials={"<|endoftext|>"}):
        tokens =self.pretokenize_text(text)
        unique_chars =[chr(i) for i in range(256)]
        unique_chars.extend(
            char for char in sorted({char for token in tokens for char in token})
            if char not in unique_chars
        )
        if "Ġ" not in unique_chars:
            unique_chars.append("Ġ")

        self.vocab= {i: char for i, char in enumerate(unique_chars)}
        self.inverse_vocab = {char: i for i, char in self.vocab.items()}

        # addin allowed special tokens
        if allowed_specials:
            for token in allowed_specials:
                if token not in self.inverse_vocab:
                    new_id =len(self.vocab)
                    self.vocab[new_id] = token
                    self.inverse_vocab[token] =new_id
        #tokenizin each pretoken into char ids
        token_id_sequences= [
            [self.inverse_vocab[char] for char in token]
            for token in tokens
        ]
        # replacin and indin frequent pairs
        for new_id in range(len(self.vocab), vocab_size):
            pair_id =self.find_freq_pair(token_id_sequences, mode="most")
            if pair_id is None:
                break
            token_id_sequences = self.replace_pair(token_id_sequences, pair_id, new_id)
            self.bpe_merges[pair_id] =new_id
        for(p0, p1), new_id in self.bpe_merges.items():
            merged_token = self.vocab[p0] + self.vocab[p1]
            self.vocab[new_id] = merged_token
            self.inverse_vocab[merged_token] = new_id

    @staticmethod
    def find_freq_pair(token_id_sequences, mode="most"):
        pairs = Counter(
            pair
            for token_ids in token_id_sequences
            for pair in zip(token_ids, token_ids[1:])
        )
        if not pairs: 
            return None
        if mode == "most":
            return max(pairs.items(), key=lambda x: x[1])[0]
        elif mode == "least":
            return min(pairs.items(), key=lambda x: x[1])[0]
        else:
            raise ValueError("Invalid mode. Choose 'most' or 'least'.")

    @staticmethod
    def replace_pair(token_id_sequences, pair_id, new_id):
        replace_sequences= []
        for token_ids in token_id_sequences:
            dq= deque(token_ids)
            replaced= []
            while dq:
                curr = dq.popleft()
                if dq and (curr, dq[0])== pair_id:
                    replaced.append(new_id)
                    dq.popleft()
                else:
                    replaced.append(curr)
            
            replace_sequences.append(replaced)
        return replace_sequences